
# Stream Customers Data From cloud Files to Delta Lake

1. Read files from cloud storage using DataStreamReader API
2. Transform the dataframe to add the following columns
    1. file_path : Cloud file path
    2. Ingestion date : Current Timestamp
3. Write the transformed data stream to Delta Lake Table


## 1. Read files using DataStreamReader API



Supported Auto Loader sources

Auto Loader can load data files from the following sources:
1. Amazon S3 (s3://)
2. Azure Data Lake Storage (ADLS)
3. Google Cloud Storage (GCS)
4. Azure Blob Storage

In [0]:
sparkAutoLoaderStreamingDF = (spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.format', 'json')
        .option('cloudFiles.schemaLocation', '/Volumes/gizmobox/landing/operational_data/customer_stream/_customer_stream_autoloader')
        .option('cloudFiles.schemaHints', 'date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP')
        .load('/Volumes/gizmobox/landing/operational_data/customer_stream/')) # here it automatically detects the schema, but we can also specify as schemaHints incase during development if we see any changes in the schema.

We can also use specific filter from which we can read files, one such filter is pathGlobalFilter : "customer_2024_*" files that start with the prefix only been picked up.
## Second, we have cloudFiles.schemaEvolutionMode, in this we have three modes

Here you go — **clean, exam/interview-ready table** 👇

| Mode                        | Stream Fails | Schema Evolves         | Handling of New Columns                                               | Typical Use Case                                         |
| --------------------------- | ------------ | ---------------------- | --------------------------------------------------------------------- | -------------------------------------------------------- |
| **addNewColumns (default)** | Yes          | Yes (only new columns) | New columns added to schema; existing column data types do not change | Silver layer where controlled evolution is allowed       |
| **rescue**                  | No           | No                     | All new/unexpected columns stored in `_rescued_data` column           | Bronze/raw ingestion with unpredictable schema           |
| **failOnNewColumns**        | Yes          | No                     | Stream fails until schema is updated or offending files are removed   | Gold layer / strict governance pipelines                 |
| **none**                    | No           | No                     | New columns ignored (unless `rescuedDataColumn` is set)               | Fixed-schema pipelines where extra fields are irrelevant |

If you want, I can also give:

* **One-line memory trick** for each mode
* **Exact Databricks config syntax** (PySpark & SQL)
* **Diagram showing Bronze → Silver → Gold usage**

Just say the word 😊



## 2. Transform the dataframe to add the following columns
    1.file_path: Cloud file path
    2.Ingestion date: Current Timestamp

In [0]:
import pyspark.sql.functions as F
sparkAutoLoaderStreamingDF = (sparkAutoLoaderStreamingDF.withColumn('file_path', F.col('_metadata.file_path'))
                            .withColumn('ingestion_date', F.current_timestamp()))


## 3. Write the transformed datastream to Delta Table

In [0]:
streaming_query = (sparkAutoLoaderStreamingDF.writeStream
                            .format('delta')
                            .trigger(once = True)
                            .option('checkpointLocation', '/Volumes/gizmobox/landing/operational_data/customer_stream/_customer_stream_autoloader_streaming')
                            .toTable("gizmobox.bronze.customers_stream_autoloader"))

In [0]:
%sql

SELECT * FROM gizmobox.bronze.customers_stream_autoloader